In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable


In [0]:
# Utilities 

In [0]:
%run /Workspace/Users/ashishbudz@gmail.com/databricks_pipeline/1_setup/utilities

In [0]:
# S3 storage folder path
base_path = "s3://sportsbar-dp-child-company-prac/orders"

In [0]:
# Setting up Widgets
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","orders","Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

landing = f"{base_path}/landing/"
processed = f"{base_path}/processed/"



In [0]:
# Reading data from the S3 storage
df = spark.read.options(header=True, inferSchema=True).csv(f"{landing}*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*","_metadata.file_name", "_metadata.file_size")

In [0]:
display(df.limit(10))

In [0]:
df.write.format("delta")\
    .option("delta.enableChangeDataFeed", True)\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")


In [0]:
bronze_df = spark.sql(f'SELECT * FROM {catalog}.{bronze_schema}.{data_source}')

In [0]:
display(bronze_df.withColumn(
    'order_qty',
    F.when(F.column('order_qty').isNull(), F.lit(1))
    .otherwise(F.column('order_qty'))
))

In [0]:
display(bronze_df.select('order_qty').summary('mean').select('order_qty').collect()[0][0])

In [0]:
avg = bronze_df.select('order_qty').summary('mean').select('order_qty').collect()[0][0]
display(bronze_df.withColumn(
    'order_qty',
    F.when(F.column('order_qty').isNull(), F.lit(avg))
    .otherwise(F.column('order_qty'))
))

In [0]:
bronze_df = bronze_df.dropna(subset = ['order_qty'])

In [0]:
display(bronze_df.select("*").where(F.column("order_qty").isNull()))

In [0]:
# Checking date formats:
display(bronze_df.select('order_placement_date').distinct())

In [0]:
# Replacing Day-of-week from date before parsing to Date object
bronze_df = bronze_df.withColumn(
    "order_placement_date",
    F.regexp_replace(F.column("order_placement_date"), "^[A-Za-z]+,\\s*", "")
).withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date(F.column("order_placement_date"), "MM/dd/yyyy"),
        F.try_to_date(F.column("order_placement_date"), "MM-dd-yyyy"),
        F.try_to_date(F.column("order_placement_date"), "dd/MM/yyyy"),
        F.try_to_date(F.column("order_placement_date"), "dd-MM-yyyy"),
        F.try_to_date(F.column("order_placement_date"), "yyyy/MM/dd"),
        F.try_to_date(F.column("order_placement_date"), "yyyy-MM-dd"),
        F.try_to_date(F.column("order_placement_date"), "MMMM dd, yyyy")
    )
)



In [0]:
import pyspark
pyspark.__version__

In [0]:
# Moving Files from 'Landing' to 'Processed' in S3 Bucket
files = dbutils.fs.ls(f'{landing}')
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f'{processed}{file_info.name}',
        True
    )

In [0]:
display(bronze_df)

In [0]:
# Test: Filtering out records where customer ID is not a valid numerical
display(bronze_df.where(F.column("customer_id").rlike("^[0-9]+$")))

In [0]:
bronze_df = bronze_df.withColumn(
    "customer_id",
    F.when(F.column("customer_id").rlike("^[0-9]+$"), F.column("customer_id"))
    .otherwise(F.lit("999999"))
)

In [0]:
# Count before dropping duplicates 
display(bronze_df.count())

In [0]:
# Dropping Duplicates
bronze_df = bronze_df.dropDuplicates(['order_id','order_placement_date','customer_id','product_id','order_qty'])

In [0]:
# Count after dropping duplicates
bronze_df.count()

In [0]:
# Converting product id to String
bronze_df = bronze_df.withColumn(
    "product_id",
    F.column("product_id").cast("string")
)

In [0]:
bronze_df.printSchema()

In [0]:
# Minimum and maximum dates
display(bronze_df.agg(
    F.min(F.column("order_placement_date")).alias("min_date"),
    F.max(F.column("order_placement_date")).alias("max_date")
))